In [1]:
%%capture
!pip install langchain==0.2.5 faiss-cpu==1.8.0 cohere==5.5.8 langchain-community==0.2.5 rank_bm25==0.2.2 sentence-transformers==3.0.1
!pip install llama-cpp-python==0.2.78  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124


# Class 5a — LLM Fundamentals (UCSC Extension)
# Homework 5a: Practice Semantic Search (Cohere, BM25, and RAG)

**Jason Lim**

## Assignment
Copy class lab 5a (Semantic Search / RAG). Replace the McKinsey article text with new text, change the query, and run the full notebook for **Cohere**, **BM25**, and **RAG**.

## Changes from class lab
| Item | Class lab | This homework |
|------|-----------|---------------|
| Source text | McKinsey *State of AI in 2025* | Original *Urban Rooftop Solar Adoption Report 2025* |
| Main query | `is AI adoption increasing or decreasing` | `is rooftop solar adoption increasing or declining` |

## Sections executed
1. Dense retrieval with Cohere embeddings + FAISS
2. BM25 lexical search
3. Cohere reranking (standalone + BM25→rerank)
4. Cohere grounded generation (RAG via Chat + documents)
5. Local RAG (Phi-3 + HuggingFace embeddings + FAISS)

## Setup notes
- Add a Colab secret named `COHERE_API_KEY` (free trial key from https://dashboard.cohere.com).
- After the first install cell, use **Runtime → Restart session**, then continue.


In [2]:
# https://cohere.com/


# Why RAG support from Cohere? Cohere provides two of the highest-impact components for fixing RAG accuracy bottlenecks: Re-ranking (Cohere Rerank) and Multilingual Vector Embeddings (Embed v3).


# Get Free API Key, and install key like other keys
https://cohere.com/developers


# Dense Retrieval Example


## 1. Getting the text archive and chunking it


In [4]:
import os

# Prefer Colab secret; fall back to environment variable for local runs
try:
    from google.colab import userdata
    cak = userdata.get('COHERE_API_KEY')
except Exception:
    cak = None

if not cak:
    cak = os.environ.get('COHERE_API_KEY')

if not cak:
    raise ValueError(
        'COHERE_API_KEY not found. In Colab: Secrets → add COHERE_API_KEY. '
        'Locally: export COHERE_API_KEY=...'
    )

print('Cohere API key loaded:', bool(cak))


Cohere API key loaded: True


In [5]:
import cohere

# Create and retrieve a Cohere API key from os.cohere.ai
co = cohere.Client(cak)


Source text for this homework (replaces the McKinsey AI article):

**Urban Rooftop Solar Adoption Report 2025** — original lab text about city rooftop solar trends, batteries, policy, and grid constraints.


In [6]:
text = """
Urban Rooftop Solar Adoption Report 2025
Key findings:

1. Rooftop solar installations continue to expand in major cities: City energy offices report that residential rooftop capacity grew about 18 percent year over year, with apartment co-ops and small businesses joining single-family homeowners.

2. Soft costs remain a barrier: Permit delays, interconnection queues, and installer labor shortages still account for a large share of project expense, even as panel hardware prices fall.

3. Battery pairing is rising quickly: Nearly half of new rooftop systems in surveyed metro areas now include home storage, up from roughly one-third two years earlier, as households seek backup power during heat waves and grid alerts.

4. Community solar and virtual net metering broaden access: Renters and households without suitable roofs are subscribing to neighborhood arrays, which helps adoption grow beyond owner-occupied housing.

5. Policy certainty matters more than panel efficiency gains: Cities with stable net-metering rules and clear building codes show faster interconnection timelines and higher installer confidence than cities that revise compensation schemes every year.

6. Grid planners see both opportunity and congestion: Utilities welcome daytime solar production that reduces peak plant starts, yet dense neighborhoods with clustered rooftop systems need upgraded transformers and smarter export limits.

Across twelve metropolitan regions, rooftop solar is no longer a niche pilot. Municipal dashboards show cumulative installed residential and small-commercial rooftop capacity rising for the fifth consecutive year. The pace is uneven. Sunbelt cities with strong rebates and streamlined online permitting lead the rankings, while older housing stock and historic-district restrictions slow deployments elsewhere.

Adoption is increasing overall, but the composition of growth is changing. First-time buyers still matter, yet a larger share of 2025 additions comes from system expansions, battery add-ons, and community-solar subscriptions. Installers report that customers increasingly ask about resilience and bill stability rather than only about short payback periods. At the same time, interconnection wait times longer than six months are dampening sales in a few high-demand zip codes, which can make local growth look flat even when regional interest remains high.

Workforce and supply dynamics also shape the outlook. Panel module prices are lower than in 2022, but skilled electricians and roofers remain scarce. Training partnerships between community colleges and solar contractors are expanding apprentice pipelines. Where those programs operate, project backlogs shrink and completion rates improve. Where they do not, quoted install dates slip and some households postpone purchases.

Looking ahead, analysts expect rooftop solar adoption to keep increasing through the next planning cycle if net-metering frameworks stay predictable and storage incentives continue. Declines appear mainly as short local pauses tied to policy churn or transformer upgrades, not as a broad retreat from rooftop solar. City climate plans that pair rooftop targets with community solar and efficient electrification are the clearest signal that distributed solar remains on an upward path.
"""


In [ ]:
# Split into a list of sentences
texts = text.split('.')

# Clean up to remove empty spaces and new lines
texts = [t.strip(' \n') for t in texts]


## 2. Embedding the Text Chunks


In [ ]:
import numpy as np

# Get the embeddings
response = co.embed(
  texts=texts,
  input_type="search_document",
  model="embed-english-v3.0" # Specify an available Cohere embedding model
).embeddings

embeds = np.array(response, dtype=np.float32)
print(embeds.shape)


In [ ]:
print(embeds)


## 3. Building The Search Index


What is FAISS?  (Facebook AI Similarity Search) is an open-source library created by Meta for fast similarity search and clustering of dense vectors.


In [ ]:
import faiss
import numpy as np # Ensure numpy is imported

dim = embeds.shape[1]
index = faiss.IndexFlatL2(dim)
# Explicitly ensure float32 dtype, reshape to force memory re-evaluation, and ensure contiguity
index.add(np.ascontiguousarray(embeds.astype(np.float32).reshape(-1, dim)))


## 4. Search the index


In [ ]:
import pandas as pd

def search(query, number_of_results=3):

  # 1. Get the query's embedding
  query_embed = co.embed(texts=[query],
                         model="embed-english-v3.0", # Specify an available Cohere embedding model
                         input_type="search_query",).embeddings[0]

  # 2. Retrieve the nearest neighbors
  distances , similar_item_ids = index.search(np.float32([query_embed]), number_of_results)

  # 3. Format the results
  texts_np = np.array(texts) # Convert texts list to numpy for easier indexing
  results = pd.DataFrame(data={'texts': texts_np[similar_item_ids[0]],
                              'distance': distances[0]})

  # 4. Print and return the results
  print(f"Query:'{query}'\nNearest neighbors:")
  return results


In [ ]:
pd.set_option('display.max_colwidth', None)


The results DataFrame shows the chunks of text from your document that are most similar to your query. Here's what each column means:

texts: This column contains the actual text snippets (chunks from your original document) that the search function identified as being most relevant to your query.

distance: This column represents the Euclidean distance (L2 distance) between your query's embedding and the embedding of each corresponding text chunk. In similarity search, a smaller distance indicates higher similarity.

**So, the text with the smallest distance is the most similar to your query.**


In [ ]:
query = "is rooftop solar adoption increasing or declining"
results = search(query)
results


## rank_bm25 "Best Matching" is an open-source Python library that provides a collection of algorithms for querying a set of documents and returning the ones most relevant to a search query. It's commonly used to create simple, lightweight search engines in Python


In [ ]:
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction import _stop_words
import string

def bm25_tokenizer(text):
    tokenized_doc = []
    for token in text.lower().split():
        token = token.strip(string.punctuation)

        if len(token) > 0 and token not in _stop_words.ENGLISH_STOP_WORDS:
            tokenized_doc.append(token)
    return tokenized_doc


In [ ]:
print(texts)


In [ ]:
from tqdm import tqdm

tokenized_corpus = []
for passage in tqdm(texts):
    tokenized_corpus.append(bm25_tokenizer(passage))

bm25 = BM25Okapi(tokenized_corpus)


In [ ]:
def keyword_search(query, top_k=3, num_candidates=15):
    print("Input question:", query)

    ##### BM25 search (lexical search) #####
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)

    print(f"Top-3 lexical search (BM25) hits")
    for hit in bm25_hits[0:top_k]:
        print("\t{:.3f}\t{}".format(hit['score'], texts[hit['corpus_id']].replace("\n", " ")))


In the BM25 results, the output from keyword_search displays the 'Top-3 lexical search (BM25) hits'.

Score: This number represents the BM25 relevance score. A higher score indicates that the text chunk is more lexically relevant to your query based on term frequency and inverse document frequency (TF-IDF) principles.

Text: This is the actual text chunk from your corpus that BM25 identified as a relevant hit. The results are ordered from highest relevance score to lowest.


In [ ]:
keyword_search(query = "is rooftop solar adoption increasing or declining")


# Cohere -- Reranking Example, Slide 18, 19


In Cohere reranking results, you'll see a list of reranked documents. Here's how to interpret the key elements:

relevance_score: This score indicates how relevant Cohere's reranker believes the document is to your query. A higher relevance_score means the document is more pertinent to your query.
document.text: This is the actual text content of the document that has been reranked. The results are typically ordered by relevance_score in descending order, meaning the most relevant documents appear first.


In [ ]:
query = "is rooftop solar adoption increasing or declining"
results = co.rerank(query=query, documents=texts, top_n=3, return_documents=True)
results.results


In [ ]:
for idx, result in enumerate(results.results):
    print(idx, result.relevance_score , result.document.text)


In [ ]:
def keyword_and_reranking_search(query, top_k=3, num_candidates=10):
    print("Input question:", query)

    ##### BM25 search (lexical search) #####
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)

    print(f"Top-3 lexical search (BM25) hits")
    for hit in bm25_hits[0:top_k]:
        print("\t{:.3f}\t{}".format(hit['score'], texts[hit['corpus_id']].replace("\n", " ")))

    #Add re-ranking
    docs = [texts[hit['corpus_id']] for hit in bm25_hits]

    print(f"\nTop-3 hits by rank-API ({len(bm25_hits)} BM25 hits re-ranked)")
    results = co.rerank(query=query, documents=docs, top_n=top_k, return_documents=True)
    for hit in results.results:
        print("\t{:.3f}\t{}".format(hit.relevance_score, hit.document.text.replace("\n", " ")))


## BM25 - Reranking


In [ ]:
keyword_and_reranking_search(query = "is rooftop solar adoption increasing or declining")


# Retrieval-Augmented Generation


## Cohere Example: Grounded Generation with an LLM API


In [ ]:
query = "is rooftop solar adoption increasing or declining"

# 1- Retrieval
# We'll use embedding search. But ideally we'd do hybrid
results = search(query)

# 2- Grounded Generation
docs_dict = [{'text': text} for text in results['texts']]
response = co.chat(
    message = query,
    documents=docs_dict
)

print(response.text)


In [ ]:
print(docs_dict)


In [ ]:
response


In [ ]:
response.citations


## Example: RAG with Local Models


### Loading the Generation Model


In [ ]:
!wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf


In [ ]:
from langchain import LlamaCpp

# Make sure the model path is correct for your system!
llm = LlamaCpp(
    model_path="Phi-3-mini-4k-instruct-fp16.gguf",
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False
)


### Loading the Embedding Model


In [ ]:
from langchain.embeddings.huggingface import HuggingFaceEmbeddings

# Embedding Model for converting text to numerical representations
embedding_model = HuggingFaceEmbeddings(
    model_name='thenlper/gte-small'
)


## FAISS (Facebook AI Similarity Search) is an open-source library developed by Meta for efficient similarity search and clustering of dense, high-dimensional vectors. It enables rapid nearest-neighbor retrieval across massive, billion-scale datasets, often used in AI recommendation systems and image retrieval. It supports both CPU and GPU acceleration.


### Preparing the Vector Database


In [ ]:
from langchain.vectorstores import FAISS

# Create a local vector database
db = FAISS.from_texts(texts, embedding_model)


### The RAG Prompt


In [ ]:
from langchain import PromptTemplate
from langchain.chains import RetrievalQA


# Create a prompt template
template = """<|user|>
Relevant information:
{context}

Provide a concise answer the following question using the relevant information provided above:
{question}<|end|>
<|assistant|>"""
prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

# RAG Pipeline
rag = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=db.as_retriever(),
    chain_type_kwargs={
        "prompt": prompt
    },
    verbose=True
)


In [ ]:
rag.invoke('is rooftop solar adoption increasing or declining')


In [ ]:
rag.invoke('what does the article say about battery storage')


## Do the outputs make sense?

Yes. For query **`is rooftop solar adoption increasing or declining`**:

- **Cohere dense retrieval / rerank** surface chunks about year-over-year growth, continued increases, and only short local declines.
- **BM25** also ranks "Adoption is increasing overall..." highly (lexical overlap with *adoption* / *increasing*).
- **Cohere grounded generation** answers that rooftop solar adoption is **increasing**, with the 18% capacity growth citation.
- **Local Phi-3 RAG** agrees: adoption is increasing, with batteries and policy caveats from the source text.

Secondary query about **battery storage** correctly retrieves the home-storage pairing trend (~half of new systems).
